# 09 · Publication-Ready Visualizations

Produces the 4 final figures for the Jupyter Book.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'
EXT = ROOT / 'data' / 'external'
FIGS = ROOT / 'output' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from plotting import duck_curve_evolution, adoption_density_heatmap, btm_vs_grid_scatter

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

panel = pd.read_parquet(PROC / 'panel_for_regression.parquet')

## Figure 1: Duck Curve Evolution (2015 vs 2023)

In [ ]:
caiso = pd.read_parquet(PROC / 'caiso_clean.parquet')
caiso['interval_start_utc'] = pd.to_datetime(caiso['interval_start_utc'], utc=True)
caiso['year'] = caiso['interval_start_utc'].dt.year
caiso['hour'] = caiso['interval_start_utc'].dt.hour

col_map = {'Solar': 'solar_mw', 'Wind': 'wind_mw', 'Load': 'demand_mw'}
caiso = caiso.rename(columns={k: v for k, v in col_map.items() if k in caiso.columns})

def get_hourly_avg(yr):
    sub = caiso[caiso['year'] == yr]
    return sub.groupby('hour')[['solar_mw', 'wind_mw', 'demand_mw']].mean().assign(
        net_load_mw=lambda d: d['demand_mw'] - d['solar_mw'] - d['wind_mw']
    ).reset_index()

h2015 = get_hourly_avg(2015)
h2023 = get_hourly_avg(2023)

fig = duck_curve_evolution(h2015, h2023)
plt.show()

## Figure 2: Adoption Density Heatmap (ZIP choropleth by year)

In [ ]:
zcta_path = EXT / 'ca_zcta.geojson'
if zcta_path.exists():
    zcta_gdf = gpd.read_file(zcta_path)
    zcta_gdf['zip_code'] = zcta_gdf['ZCTA5CE20'].astype(str).str.zfill(5)
    
    # Join EOY capacity by year
    eoy = panel[panel['month'] == 12][['zip_code', 'year', 'btm_capacity_kw']]
    geo_panel = zcta_gdf.merge(eoy, on='zip_code', how='inner')
    
    fig = adoption_density_heatmap(geo_panel, year_col='year', capacity_col='btm_capacity_kw')
    plt.show()
else:
    print(f'ZCTA shapefile not found at {zcta_path} — skipping choropleth')

## Figure 3: Curtailment Intensity Map

In [ ]:
# Curtailment is system-level from CAISO; show as time-series annotation on map
# Use panel's curtailment_days_per_month averaged per ZIP (constant across ZIPs since it's system-level)
# Overlay as inset on BTM density map or note the limitation clearly

if zcta_path.exists():
    zip_curtailment = panel.groupby('zip_code')['curtailment_days_per_month'].mean().reset_index()
    geo_curt = zcta_gdf.merge(zip_curtailment, on='zip_code', how='inner')
    zip_btm = panel[panel['year'] == 2023].groupby('zip_code')['btm_capacity_kw'].mean().reset_index()
    geo_curt = geo_curt.merge(zip_btm, on='zip_code', how='left')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    geo_curt.plot(column='btm_capacity_kw', cmap='YlOrRd', legend=True, ax=axes[0],
                  linewidth=0.1, edgecolor='white')
    axes[0].set_title('BTM Capacity (kW, 2023)', fontsize=12)
    axes[0].axis('off')
    
    geo_curt.plot(column='curtailment_days_per_month', cmap='Blues', legend=True, ax=axes[1],
                  linewidth=0.1, edgecolor='white')
    axes[1].set_title('Avg Curtailment Days/Month (CAISO system-level)', fontsize=12)
    axes[1].axis('off')
    
    fig.suptitle('BTM Adoption Density & Curtailment Intensity', fontsize=14)
    plt.tight_layout()
    fig.savefig(FIGS / 'curtailment_intensity_map.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGS / 'curtailment_intensity_map.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Skipping curtailment map — ZCTA shapefile needed')

## Figure 4: BTM Capacity vs. Grid Stress Scatter

In [ ]:
fig = btm_vs_grid_scatter(panel, outcome='ramp_magnitude_mwh')
plt.show()